<a href="https://colab.research.google.com/github/janithcyapa/Statistical-Learning-e20452/blob/main/Assignments/Assignment_05_Gaussian_Process_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment : Gaussian Process Regression & Linear Regression**
### ME526 : Introduction to Statistical Learning
---
> YAPA W.S.P.Y.J.C.
>
> E/20/452
---

In [ ]:
!pip install git+https://github.com/janithcyapa/Statistical-Learning-e20452.git

## **Gaussian Process Regression**

Modeling both the Heating Load (Y1) and Cooling Load (Y2) as a single parameter Gaussian Process need to handle two dependent thermal loads simultaneously, we therefore General Multivariate Formulation is used here.

Instead of two independent scalar processes, define  observation $Y_g$ at any building configuration g as a vector in $\mathbb{R}^2$,
$$Y_g =X_g + ν_g$$

where $X_g ∈ R_2$ is the latent clean thermal load, and $ν_g ∼ N(0,Σν​)$ is the noise vector.
\
\
If we have $n$ building observations, our stacked observation vector $\mathscr{Y}_n$ is in $\mathbb{R}^{2n}$. The joint marginal distribution becomes:


$$\mathscr{Y}_n \sim \mathscr{N} \left( \mu_{Y,n}, K_n + I_n \otimes \Sigma_\nu \right)$$

Here, the covariance matrix $K_n$ is built from a matrix-valued kernel $\kappa(g, g') \in \mathbb{R}^{2 \times 2}$. The diagonal blocks represent the variance of heating and cooling loads independently, while the off-diagonal elements capture the cross-correlation between them.

To estimate the thermal loads $X_g$ for a new building configuration, the predictive mean relies on the block matrix inversion,

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + K_{g,n} \left( K_n + I_n \otimes \Sigma_\nu \right)^{-1} \left( y_n - \mu_{Y,n} \right)$$

By modeling them together, the GP leverages the shared architectural features (X1 to X8) and the intrinsic thermodynamic coupling between heating and cooling to create a more robust predictive distribution.

In [20]:
import pandas as pd
import kagglehub
import plotly.graph_objects as go
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [19]:
# Load the data
path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
df = pd.read_csv(path + "/ENB2012_data.csv").dropna()

# Prepare X (Features 1-8) and Y (Targets Y1, Y2)
X = df.iloc[:, :8]
y = df[['Y1', 'Y2']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features (Crucial for distance-based kernels)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define Kernel and Model
# Variance * RBF(length_scale) + Noise
kernel = C(1.0, (1e-3, 1e3)) * RBF(10, (1e-2, 1e2)) + WhiteKernel(noise_level=1, noise_level_bounds=(1e-5, 1e1))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)

# Fit and Predict
gp.fit(X_train_scaled, y_train)
y_pred, y_std = gp.predict(X_test_scaled, return_std=True)

print(f"Optimized Kernel: {gp.kernel_}")
print(f"R2 Score (Heating): {r2_score(y_test['Y1'], y_pred[:, 0]):.4f}")
print(f"R2 Score (Cooling): {r2_score(y_test['Y2'], y_pred[:, 1]):.4f}")

# Plotting Heating Load vs Prediction
fig = go.Figure()

# Ideal fit line
fig.add_trace(go.Scatter(x=[y_test['Y1'].min(), y_test['Y1'].max()],
                         y=[y_test['Y1'].min(), y_test['Y1'].max()],
                         mode='lines', line=dict(color='gray', dash='dash'), name='Ideal Fit'))

# Actual predictions
fig.add_trace(go.Scatter(x=y_test['Y1'], y=y_pred[:, 0],
                         mode='markers', marker=dict(color='cyan', size=6, opacity=0.7),
                         name='Predictions (Y1)'))

fig.update_layout(title='GPR: Actual vs Predicted Heating Load',
                  xaxis_title='Actual Heating Load (Y1)',
                  yaxis_title='Predicted Heating Load',
                  template='plotly_dark')
fig.show()

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Optimized Kernel: 1.92**2 * RBF(length_scale=1.82) + WhiteKernel(noise_level=0.00524)
R2 Score (Heating): 0.9973
R2 Score (Cooling): 0.9816


## **Linear Regression**


To predict the 'predicted_energy_demand' ($Y$) using a linear relationship of selected building environment parameters, here formulate the problem using Multivariate Linear Regression.

Instead of relying on a single variable, I have define input $X_i$ as a feature vector encompassing the selected drivers for a given building configuration $i$. The scalar predicted energy demand $Y_i$ is modeled as a linear combination of these features,

$$Y_i = \beta X_i + \beta_0 + \nu_i$$

where $\beta$ is the weight matrix, $\beta_0$ is the intercept, and $\nu_i \sim \mathscr{N}(0, \Sigma_\nu)$ represents the additive Gaussian noise.

If we have $n$ observations in our dataset, I can represent this compactly by defining an augmented input vector $\widetilde{x}_i = [x_i^T, 1]^T$. By stacking these into an augmented design matrix $\widetilde{X}$, stacking the outputs into $Y$, and combining the parameters into $W = [\beta^T, \beta_0^T]^T$, the full system becomes,

$$Y = \widetilde{X} W + N$$

Assuming the observation noise is independent across samples and isotropic (i.e., $\Sigma_\nu = \sigma^2 I_n$), maximizing the log-marginal likelihood of this Gaussian distribution mathematically simplifies to minimizing the sum of squared residuals. Therefore, assuming $\widetilde{X}^T \widetilde{X}$ is invertible, the Maximum Likelihood Estimator (MLE) for the parameter matrix is given by the normal equation,

$$\widehat{W}_{\mathrm{MLE}} = (\widetilde{X}^T \widetilde{X})^{-1} \widetilde{X}^T Y$$

By calculating $\widehat{W}_{\mathrm{MLE}}$, the linear regression directly identifies the constant $\beta$ coefficients. This provides a deterministic mapping that shows exactly how the superposition of individual sub-system parameters drives the overall predicted energy demand.

In [21]:
# Load the data
path_lr = kagglehub.dataset_download("programmer3/green-building-multi-source-environment-dataset")
df2 = pd.read_csv(path_lr + "/green_building_dataset.csv").dropna()

# Parameter Selection based on thermal and internal load dynamics
features = ['ventilation_rate', 'equipment_load', 'heating_energy', 'cooling_energy', 'occupancy']
X2 = df2[features]
y2 = df2['predicted_energy_demand']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

# Fit Model
lr = LinearRegression()
lr.fit(X2_train, y2_train)

# Predict and Evaluate
y2_pred = lr.predict(X2_test)

print("Linear Regression Coefficients:")
for feat, coef in zip(features, lr.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"Intercept: {lr.intercept_:.4f}")
print(f"R2 Score: {r2_score(y2_test, y2_pred):.4f}")
print(f"MSE: {mean_squared_error(y2_test, y2_pred):.4f}")

# Plotting (Plotly Dark)
fig2 = go.Figure()

# Ideal fit line
fig2.add_trace(go.Scatter(x=[y2_test.min(), y2_test.max()],
                          y=[y2_test.min(), y2_test.max()],
                          mode='lines', line=dict(color='gray', dash='dash'), name='Ideal Fit'))

# Predictions
fig2.add_trace(go.Scatter(x=y2_test, y=y2_pred,
                          mode='markers', marker=dict(color='magenta', size=6, opacity=0.7),
                          name='LR Predictions'))

fig2.update_layout(title='LR: Actual vs Predicted Energy Demand',
                   xaxis_title='Actual Energy Demand',
                   yaxis_title='Predicted Energy Demand',
                   template='plotly_dark')
fig2.show()

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.
Linear Regression Coefficients:
  ventilation_rate: 0.0485
  equipment_load: 0.0959
  heating_energy: 0.2414
  cooling_energy: 0.2520
  occupancy: 0.0443
Intercept: 8.0652
R2 Score: 0.7760
MSE: 20.9101
